# Canopy and Ground: site/year labeling

This notebook replaces the repeated processing in `Canopy_2015.ipynb` through `Canopy_2018.ipynb` and `Ground_2015.ipynb` through `Ground_2018.ipynb`. Select sites and years below; the shared `scripts/` package handles feature selection, cleaning, feature engineering, and the three fault labels.

Run in Google Colab after `Data_Overview.ipynb` has generated the merged input files. Code is cloned into the runtime; all input data and exported results live on Google Drive. The original notebooks remain available as references.


## 1. Clone the repository and import shared functions

The repository must be public. An existing runtime checkout is reused; restart the runtime to obtain a fresh clone after repository updates. No data is downloaded from GitHub.


In [ ]:
import gc
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/dulkusim/pv-timeseries-fault-forecasting.git"
repo_path = Path("/content/pv-timeseries-fault-forecasting")
if not repo_path.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(repo_path)],
        check=True,
    )

labeling_dir = repo_path / "notebooks" / "Data Labeling"
if not (labeling_dir / "scripts" / "labeling.py").is_file():
    raise FileNotFoundError(f"Shared scripts not found in {labeling_dir}")
if str(labeling_dir) not in sys.path:
    sys.path.insert(0, str(labeling_dir))

import pandas as pd
from IPython.display import display
from scripts.config import CANOPY, GROUND
from scripts.io import get_data_root, load_site_year, validate_site_year
from scripts.feature_selection import (
    rtd_report, detect_faulty_rtds, correlation_report,
)
from scripts.cleaning import missing_report
from scripts.inverter import ORIGINAL_INVERTER_POLICY, REVISED_INVERTER_POLICY
from scripts.labeling import run_labeling, summarize


## 2. Mount Google Drive and select datasets

Inputs: `MyDrive/NIST_processed/onemin-Canopy-2015.csv` (and the corresponding site/year files).

Outputs: `MyDrive/NIST_Labels/canopy_2015.csv` (and the corresponding site/year files), plus small combined label and processing summaries in the same directory. Final CSVs retain `TIMESTAMP` as their index column, as in the original final exports. Intermediate CSVs are no longer saved and reloaded.

The defaults process both sites for 2015–2018. For a single dataset, use `SITE_NAMES = ("canopy",)` and `YEARS = (2015,)`. Existing outputs cause a preflight error unless `OVERWRITE` is enabled. Diagnostics report missing values, RTD means, candidate faulty RTDs, and strong electrical correlations; these reports do not change the pipeline's established column removal policy.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
os.environ["DATA_ROOT"] = "/content/drive/MyDrive"
root = get_data_root()

SITE_NAMES = ("canopy", "ground")
YEARS = (2015, 2016, 2017, 2018)
SHOW_DIAGNOSTICS = True
OVERWRITE = False

# Preserve the original notebooks' inverter rules.
# Set to REVISED_INVERTER_POLICY to deliberately use the revised rules.
INVERTER_POLICY = ORIGINAL_INVERTER_POLICY

SITES = {"canopy": CANOPY, "ground": GROUND}
if not SITE_NAMES or len(set(SITE_NAMES)) != len(SITE_NAMES):
    raise ValueError("Select at least one site, without duplicates")
if set(SITE_NAMES) - set(SITES):
    raise ValueError(f"Unknown sites: {set(SITE_NAMES) - set(SITES)}")
if not YEARS or len(set(YEARS)) != len(YEARS):
    raise ValueError("Select at least one year, without duplicates")
if set(YEARS) - {2015, 2016, 2017, 2018}:
    raise ValueError("Supported years are 2015–2018")

output_dir = root / "NIST_Labels"
jobs = [(SITES[name], year) for name in SITE_NAMES for year in YEARS]
summary_path = output_dir / "label_summary.csv"
processing_path = output_dir / "processing_summary.csv"

# Check all inputs and outputs before starting expensive processing.
input_paths = [
    root / "NIST_processed" / site.raw_file_pattern.format(year=year)
    for site, year in jobs
]
missing_inputs = [str(path) for path in input_paths if not path.is_file()]
if missing_inputs:
    raise FileNotFoundError("Missing merged input files:\n" + "\n".join(missing_inputs))
planned_outputs = [
    output_dir / f"{site.name}_{year}.csv" for site, year in jobs
] + [summary_path, processing_path]
existing = [str(path) for path in planned_outputs if path.exists()]
if existing and not OVERWRITE:
    raise FileExistsError(
        "Outputs already exist. Set OVERWRITE = True to replace them:\n"
        + "\n".join(existing)
    )
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Selected datasets: {[(site.name, year) for site, year in jobs]}")
print(f"Output directory: {output_dir}")
print(f"Inverter policy: {INVERTER_POLICY}")


## 3. Shared processing pipeline

`load_site_year` parses and sorts timestamps and rejects empty data, missing timestamps, and timestamps outside the selected year. `run_labeling` then performs:

1. Secondary/redundant feature removal, site-specific RTD removal, the shared missingness policy (remove `WindSpeedAve_ms`), and bounds cleaning of string current/power.
2. Operating mask, fused POA irradiance, median module temperature, and PR evaluation mask.
3. Temperature-corrected expected power, weighted rolling performance ratios, and `label_power`.
4. String aggregates, voltage proxies, irradiance normalization, baseline deviations, and `label_string`.
5. Inverter diagnostics and `label_inverter`.

Canopy uses its configured east/west irradiance sensors, 243 kW DC rating, and strings 2–6 for string fault decisions. Ground uses its configured single POA sensor, 271 kW DC rating, and all seven strings. Both use the 260 kW AC ceiling. The notebook passes each site's configured core strings explicitly.

The original inverter policy uses AC power below 0.05 kW for power shutdown, missing/nonpositive status for status shutdown, DC power above 1 kW for efficiency evaluation, and positive reported fault codes. The revised policy instead uses DC power below 1 kW, also flags status above 5, requires DC power above 5 kW for efficiency evaluation, and treats any nonzero reported fault code as a fault.

String baselines are fitted separately on each selected site's annual evaluable data, matching the exploratory workflow. These annual baselines use the full year; for later forecasting experiments, fit baselines on training data only and pass them through `run_labeling(..., baselines=...)`.

Rolling windows in the scripts are observation counts, assuming one-minute sampling. Gaps and duplicate timestamps are reported below; they are not filled or removed automatically.


In [ ]:
def process_site_year(site, year):
    """Process and export one dataset; return only small summary tables."""
    print(f"\n=== {site.name.upper()} {year} ===")
    df = load_site_year(site, year)
    input_shape = df.shape
    duplicate_timestamps = int(df.index.duplicated().sum())
    nonminute_steps = int(
        df.index.to_series().diff().dropna().ne(pd.Timedelta(minutes=1)).sum()
    )
    print(f"Input shape: {input_shape}; range: {df.index.min()} to {df.index.max()}")
    print(f"Duplicate timestamps: {duplicate_timestamps}")
    print(f"Timestamp steps different from one minute: {nonminute_steps}")

    if SHOW_DIAGNOSTICS:
        print("Missing measurements (before preprocessing):")
        display(missing_report(df))
        print("RTD mean temperatures:")
        display(rtd_report(df))
        print(f"Diagnostic RTD candidates: {detect_faulty_rtds(df)}")
        print(f"Configured RTDs removed: {site.rtd_drop}")
        print("Electrical feature correlations with |r| >= 0.9:")
        display(correlation_report(df))

    labeled = run_labeling(
        df,
        site,
        core_strings=site.core_strings,
        inverter_policy=INVERTER_POLICY,
        missing_power_label="MISSING_DATA",
    )
    del df
    gc.collect()

    validate_site_year(labeled, year)
    if len(labeled) != input_shape[0]:
        raise ValueError("Labeling unexpectedly changed the row count")
    if labeled.columns.duplicated().any():
        raise ValueError("Labeling produced duplicate column names")

    label_summary = summarize(labeled)
    label_summary.insert(0, "year", year)
    label_summary.insert(0, "site", site.name)
    display(label_summary)

    output = output_dir / f"{site.name}_{year}.csv"
    if output.exists() and not OVERWRITE:
        raise FileExistsError(output)
    labeled.to_csv(output, index=True, index_label="TIMESTAMP")
    print(f"Saved {labeled.shape} -> {output}")

    processing_summary = {
        "site": site.name,
        "year": year,
        "rows": len(labeled),
        "input_columns": input_shape[1],
        "output_columns": labeled.shape[1],
        "first_timestamp": labeled.index.min(),
        "last_timestamp": labeled.index.max(),
        "duplicate_timestamps": duplicate_timestamps,
        "nonminute_steps": nonminute_steps,
        "inverter_policy": repr(INVERTER_POLICY),
        "output_path": str(output),
    }
    # No annual DataFrame is retained between jobs.
    del labeled
    gc.collect()
    return label_summary, processing_summary


## 4. Run selected datasets and save summaries

Only one site/year is processed at a time. The pipeline can still hold several copies of that dataset while calculating features, so a 500 MB CSV may require substantially more Colab RAM than its disk size. Reduce the selection to a single site/year when inspecting a dataset.

Errors stop the run. Already completed exports remain on Drive. To resume without overwriting those exports, select only unfinished jobs. The combined summary CSVs describe the jobs selected in this run; when overwriting them with a smaller selection, previous summary rows are replaced.


In [ ]:
label_summaries = []
processing_records = []

for site, year in jobs:
    label_summary, processing_summary = process_site_year(site, year)
    label_summaries.append(label_summary)
    processing_records.append(processing_summary)

all_label_summaries = pd.concat(label_summaries, ignore_index=True)
processing_overview = pd.DataFrame(processing_records)
all_label_summaries.to_csv(summary_path, index=False)
processing_overview.to_csv(processing_path, index=False)

display(processing_overview)
display(all_label_summaries)
print(f"Saved label summaries: {summary_path}")
print(f"Saved processing summary: {processing_path}")
